# 🎧 YouTube Mixtape Automation

**End-to-end pipeline:** Upload tracks → Smooth fade mixtape → Background video → YouTube description → MLflow tracking → ZIP artifacts

---
### 📁 Expected Colab folder structure (upload manually before running)
```
/content/
├── audio/          ← upload your .mp3 / .wav / .flac / .m4a tracks here
│   ├── track1.mp3
│   └── ...
├── images/         ← upload your background image here
│   └── cover.jpg
└── output/         ← auto-created; all outputs land here
```
---

## ⚙️ Section 0 — Environment Setup & Secret Loading

In [ ]:
import os
from google.colab import userdata

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ updated

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

## 📦 Section 1 — Install Dependencies

In [ ]:
!pip install -q pydub pillow mlflow pymongo ipywidgets
!apt-get install -qq ffmpeg
print('✅ All dependencies installed.')

## 🗂️ Section 2 — Config (mirrors `config.py`)

In [ ]:
import os

# ── Hardcoded Colab paths ─────────────────────────────────────────────────────
AUDIO_DIR   = '/content/audio'      # ← upload your tracks here
IMAGE_DIR   = '/content/images'     # ← upload your cover image here
OUTPUT_DIR  = '/content/output'     # ← all generated files land here

ALLOWED_AUDIO_EXT = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')

# ── Mixtape settings ──────────────────────────────────────────────────────────
TRANSITION_MS  = 6000          # crossfade duration in milliseconds
OUTPUT_MP3     = 'mixtape.mp3'
OUTPUT_MP4     = 'mixtape_vid.mp4'
MIXTAPE_NAME   = 'Afro House Summer Mix'
GENRE          = 'Afro House'
COVER_IMAGE    = 'cover.jpg'   # filename inside IMAGE_DIR

# ── MLflow experiment ─────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = 'youtube-mixtape-automation'

# ── Create dirs ───────────────────────────────────────────────────────────────
for d in [AUDIO_DIR, IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Config loaded.')
print(f'   Audio dir  : {AUDIO_DIR}')
print(f'   Image dir  : {IMAGE_DIR}')
print(f'   Output dir : {OUTPUT_DIR}')
print(f'   Transition : {TRANSITION_MS} ms')

## 🛠️ Section 3 — Utilities (mirrors `utils.py`)

In [ ]:
import uuid, threading, json

# ── Simple in-memory job store ────────────────────────────────────────────────
JOB_STORE      = {}
JOB_STORE_LOCK = threading.Lock()

def new_job():
    job_id = uuid.uuid4().hex
    with JOB_STORE_LOCK:
        JOB_STORE[job_id] = {'status': 'pending', 'result': None, 'error': None}
    return job_id

def set_job_status(job_id, status, result=None, error=None):
    with JOB_STORE_LOCK:
        JOB_STORE[job_id].update({'status': status, 'result': result, 'error': error})

def get_job(job_id):
    return JOB_STORE.get(job_id)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Utility functions ready.')

## 🔍 Section 4 — Discover Audio Tracks

In [ ]:
track_paths = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(ALLOWED_AUDIO_EXT)
])

if not track_paths:
    raise FileNotFoundError(
        f'❌ No audio files found in {AUDIO_DIR}.\n'
        'Please upload your tracks to /content/audio/ and re-run this cell.'
    )

print(f'✅ Found {len(track_paths)} track(s):')
for p in track_paths:
    print(f'   • {os.path.basename(p)}')

## 🎵 Section 5 — Audio Mixing with Smooth Fades (mirrors `audio.py`)

In [ ]:
from pydub import AudioSegment
import time

def smooth_fade_mixtape_from_files(file_paths, output_filename='mixtape.mp3', transition_ms=6000):
    """
    Blends a list of audio files into a single mixtape MP3
    using crossfade with low-pass filtering on transitions.

    Parameters
    ----------
    file_paths      : list of str — ordered track paths
    output_filename : str         — output MP3 filename
    transition_ms   : int         — crossfade duration in ms

    Returns
    -------
    str : path to the exported MP3
    """
    mixtape = None
    for i, filepath in enumerate(file_paths):
        print(f'   [{i+1}/{len(file_paths)}] Loading: {os.path.basename(filepath)}')
        song = AudioSegment.from_file(filepath)
        song = song.set_channels(2).set_frame_rate(44100)  # normalize stereo @ 44.1 kHz
        if mixtape is None:
            mixtape = song
        else:
            overlap    = min(transition_ms, len(song), len(mixtape))
            outgoing   = mixtape[-overlap:].fade_out(overlap).low_pass_filter(4000)
            incoming   = song[:overlap].fade_in(overlap).low_pass_filter(4000)
            transition = outgoing.overlay(incoming)
            mixtape    = mixtape[:-overlap] + transition + song[overlap:]
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, output_filename)
    mixtape.export(out_path, format='mp3')
    return out_path


print('🎵 Starting mixtape creation...')
t0     = time.time()
job_id = new_job()
set_job_status(job_id, 'running')

try:
    mixtape_path = smooth_fade_mixtape_from_files(
        track_paths,
        output_filename=OUTPUT_MP3,
        transition_ms=TRANSITION_MS
    )
    elapsed = round(time.time() - t0, 1)
    set_job_status(job_id, 'completed', result=mixtape_path)
    print(f'\n✅ Mixtape created in {elapsed}s → {mixtape_path}')
except Exception as e:
    set_job_status(job_id, 'failed', error=str(e))
    raise

print(f'   Job status : {get_job(job_id)}')

### 💾 Checkpoint 1 — Save Mixtape Metadata

In [ ]:
checkpoint_1 = {
    'checkpoint'   : 'mixtape_audio',
    'job_id'       : job_id,
    'status'       : get_job(job_id)['status'],
    'mixtape_path' : mixtape_path,
    'tracks'       : [os.path.basename(p) for p in track_paths],
    'transition_ms': TRANSITION_MS,
    'elapsed_sec'  : elapsed
}

chk1_path = os.path.join(OUTPUT_DIR, 'checkpoint_1_audio.json')
with open(chk1_path, 'w') as f:
    json.dump(checkpoint_1, f, indent=2)

print(f'✅ Checkpoint 1 saved → {chk1_path}')
print(json.dumps(checkpoint_1, indent=2))

## 📝 Section 6 — YouTube Description Generator (mirrors `description.py`)

In [ ]:
def generate_youtube_description_with_timestamps(
    track_paths, mixtape_name='Mixtape', genre='Mix', start_time_sec=0
):
    """
    Generates a YouTube-ready description with a timestamped tracklist.

    Parameters
    ----------
    track_paths    : list of str — audio file paths in playback order
    mixtape_name   : str         — title shown at top of description
    genre          : str         — genre label
    start_time_sec : int         — offset for first timestamp (usually 0)

    Returns
    -------
    str : full YouTube description text
    """
    if not track_paths:
        raise ValueError('No tracks provided')
    description  = f'🔥 {mixtape_name} 🔥\nGenre: {genre}\n\n🎵 Tracklist:\n'
    current_time = start_time_sec
    for path in track_paths:
        audio        = AudioSegment.from_file(path)
        duration_sec = len(audio) // 1000
        minutes      = current_time // 60
        seconds      = current_time % 60
        timestamp    = f'{minutes:02d}:{seconds:02d}'
        name         = os.path.splitext(os.path.basename(path))[0]
        description += f'{timestamp} - {name}\n'
        current_time += duration_sec
    description += '\n💽 Download/Listen links:\nYou can find these tracks online.\n'
    description += '\n🎧 Follow for more mixes!\n\n'
    hashtags = ['#Mixtape', '#DJMix', '#HouseMusic', '#MusicMix', '#AfroHouse', '#ElectronicMusic']
    description += ' '.join(hashtags)
    return description


print('📝 Generating YouTube description...')
youtube_description = generate_youtube_description_with_timestamps(
    track_paths, mixtape_name=MIXTAPE_NAME, genre=GENRE
)

desc_path = os.path.join(OUTPUT_DIR, 'youtube_description.txt')
with open(desc_path, 'w') as f:
    f.write(youtube_description)

print(f'✅ Description saved → {desc_path}')
print('\n' + '-'*60)
print(youtube_description)
print('-'*60)

### 💾 Checkpoint 2 — Save Description Metadata

In [ ]:
checkpoint_2 = {
    'checkpoint'      : 'youtube_description',
    'mixtape_name'    : MIXTAPE_NAME,
    'genre'           : GENRE,
    'track_count'     : len(track_paths),
    'description_path': desc_path
}

chk2_path = os.path.join(OUTPUT_DIR, 'checkpoint_2_description.json')
with open(chk2_path, 'w') as f:
    json.dump(checkpoint_2, f, indent=2)

print(f'✅ Checkpoint 2 saved → {chk2_path}')

## 🎬 Section 7 — Video Generator (mirrors `video.py`)

In [ ]:
import subprocess
from PIL import Image

def make_video_from_audio(
    image_path, audio_path,
    output_filename='mixtape_vid.mp4',
    video_resolution=(1280, 720),
    fps=1, preset='ultrafast'
):
    """
    Creates a YouTube-ready MP4 by looping a still image over the mixtape audio.

    Parameters
    ----------
    image_path       : str   — path to cover/background image
    audio_path       : str   — path to the mixtape MP3
    output_filename  : str   — output MP4 filename
    video_resolution : tuple — (width, height) in pixels
    fps              : int   — frames per second
    preset           : str   — ffmpeg encoding preset

    Returns
    -------
    str : path to the exported MP4
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'Image not found: {image_path}')
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f'Audio not found: {audio_path}')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    temp_img = os.path.join(OUTPUT_DIR, 'temp_resized_image.jpg')
    img = Image.open(image_path).convert('RGB')
    img = img.resize(video_resolution)
    img.save(temp_img)
    out_path = os.path.join(OUTPUT_DIR, output_filename)
    cmd = [
        'ffmpeg', '-y',
        '-loop', '1',
        '-i', temp_img,
        '-i', audio_path,
        '-c:v', 'libx264',
        '-preset', preset,
        '-tune', 'stillimage',
        '-r', str(fps),
        '-c:a', 'aac',
        '-b:a', '192k',
        '-shortest',
        out_path
    ]
    print('🎬 Running ffmpeg...')
    subprocess.run(cmd, check=True, capture_output=True)
    os.remove(temp_img)
    return out_path


image_path = os.path.join(IMAGE_DIR, COVER_IMAGE)

if not os.path.exists(image_path):
    print(f'⚠️  Cover image not found at {image_path}.')
    print('   Upload cover.jpg to /content/images/ and re-run.')
    video_path  = None
    vid_elapsed = None
    vid_job_id  = new_job()
    set_job_status(vid_job_id, 'skipped')
else:
    print(f'🎬 Creating video: {image_path} + {mixtape_path}')
    t0         = time.time()
    vid_job_id = new_job()
    set_job_status(vid_job_id, 'running')
    try:
        video_path = make_video_from_audio(
            image_path=image_path,
            audio_path=mixtape_path,
            output_filename=OUTPUT_MP4
        )
        vid_elapsed = round(time.time() - t0, 1)
        set_job_status(vid_job_id, 'completed', result=video_path)
        print(f'\n✅ Video created in {vid_elapsed}s → {video_path}')
    except Exception as e:
        set_job_status(vid_job_id, 'failed', error=str(e))
        raise

### 💾 Checkpoint 3 — Save Video Metadata

In [ ]:
checkpoint_3 = {
    'checkpoint' : 'mixtape_video',
    'job_id'     : vid_job_id,
    'status'     : get_job(vid_job_id)['status'],
    'video_path' : video_path,
    'cover_image': image_path if video_path else None,
    'elapsed_sec': vid_elapsed
}

chk3_path = os.path.join(OUTPUT_DIR, 'checkpoint_3_video.json')
with open(chk3_path, 'w') as f:
    json.dump(checkpoint_3, f, indent=2)

print(f'✅ Checkpoint 3 saved → {chk3_path}')

## 📊 Section 8 — MLflow Tracking (Params + Artifacts)

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f'📊 MLflow experiment : {MLFLOW_EXPERIMENT}')
print(f'   Tracking URI      : {os.environ["MLFLOW_TRACKING_URI"]}')

with mlflow.start_run(run_name=MIXTAPE_NAME.replace(' ', '_') + '_run') as run:

    # ── Parameters ───────────────────────────────────────────────────────────
    mlflow.log_params({
        'mixtape_name'  : MIXTAPE_NAME,
        'genre'         : GENRE,
        'transition_ms' : TRANSITION_MS,
        'track_count'   : len(track_paths),
        'output_mp3'    : OUTPUT_MP3,
        'output_mp4'    : OUTPUT_MP4,
        'cover_image'   : COVER_IMAGE,
    })

    # ── Metrics ──────────────────────────────────────────────────────────────
    mlflow.log_metric('audio_elapsed_sec', elapsed)
    if vid_elapsed:
        mlflow.log_metric('video_elapsed_sec', vid_elapsed)

    # ── Artifacts ────────────────────────────────────────────────────────────
    print('   Logging artifacts...')
    mlflow.log_artifact(mixtape_path, artifact_path='audio')
    print(f'   ✔ MP3  → {mixtape_path}')

    mlflow.log_artifact(desc_path, artifact_path='description')
    print(f'   ✔ TXT  → {desc_path}')

    if video_path and os.path.exists(video_path):
        mlflow.log_artifact(video_path, artifact_path='video')
        print(f'   ✔ MP4  → {video_path}')

    for chk in [chk1_path, chk2_path, chk3_path]:
        mlflow.log_artifact(chk, artifact_path='checkpoints')
        print(f'   ✔ CHK  → {chk}')

    run_id = run.info.run_id
    print(f'\n✅ MLflow run complete. Run ID: {run_id}')

## 🗄️ Section 9 — MongoDB — Save Run Metadata

In [ ]:
from pymongo import MongoClient
from datetime import datetime

mongo_url = os.environ.get('MONGO_DB_URL', '')

if mongo_url:
    try:
        client = MongoClient(mongo_url, serverSelectionTimeoutMS=5000)
        db     = client['mixtape_automation']
        col    = db['runs']
        doc = {
            'run_id'        : run_id,
            'mixtape_name'  : MIXTAPE_NAME,
            'genre'         : GENRE,
            'transition_ms' : TRANSITION_MS,
            'track_count'   : len(track_paths),
            'tracks'        : [os.path.basename(p) for p in track_paths],
            'mixtape_path'  : mixtape_path,
            'video_path'    : video_path,
            'desc_path'     : desc_path,
            'audio_elapsed' : elapsed,
            'video_elapsed' : vid_elapsed,
            'created_at'    : datetime.utcnow().isoformat()
        }
        result = col.insert_one(doc)
        print(f'✅ MongoDB: document inserted — _id = {result.inserted_id}')
    except Exception as e:
        print(f'⚠️  MongoDB insert failed: {e}')
else:
    print('⚠️  MONGO_DB_URL not set — skipping MongoDB save.')

## 📋 Section 10 — Summary Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display

rows = [
    ('🎵 Mixtape MP3',   mixtape_path),
    ('📝 Description',   desc_path),
    ('🎬 Video MP4',     video_path or '— skipped (no cover image)'),
    ('📊 MLflow Run ID', run_id),
    ('🔢 Tracks Mixed',  str(len(track_paths))),
    ('⏱  Audio Time',    f'{elapsed}s'),
    ('⏱  Video Time',    f'{vid_elapsed}s' if vid_elapsed else '—'),
]

items = []
for label, value in rows:
    lbl = widgets.HTML(f"<b style='color:#4A90D9;min-width:160px;display:inline-block'>{label}</b>")
    val = widgets.Label(value=str(value))
    items.append(widgets.HBox([lbl, val], layout=widgets.Layout(gap='16px')))

box = widgets.VBox(
    [widgets.HTML('<h3>✅ Pipeline Complete — Summary</h3>')] + items,
    layout=widgets.Layout(border='1px solid #ccc', padding='16px', border_radius='8px')
)
display(box)

## 📦 Section 11 — ZIP All Artifacts & Download

In [ ]:
import zipfile
from google.colab import files

zip_name = f"mixtape_artifacts_{MIXTAPE_NAME.replace(' ', '_')}.zip"
zip_path = os.path.join('/content', zip_name)

artifacts_to_zip = [
    mixtape_path,
    desc_path,
    chk1_path,
    chk2_path,
    chk3_path,
]
if video_path and os.path.exists(video_path):
    artifacts_to_zip.append(video_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in artifacts_to_zip:
        if fp and os.path.exists(fp):
            arcname = os.path.basename(fp)
            zf.write(fp, arcname)
            print(f'   + {arcname}')

print(f'\n✅ ZIP created → {zip_path}')
print('⬇️  Downloading...')
files.download(zip_path)

---
## 🎉 Done!

| Output | Location |
|--------|----------|
| 🎵 Mixtape MP3 | `/content/output/mixtape.mp3` |
| 🎬 Video MP4 | `/content/output/mixtape_vid.mp4` |
| 📝 YouTube Description | `/content/output/youtube_description.txt` |
| 💾 Checkpoints | `/content/output/checkpoint_*.json` |
| 📊 MLflow Run | DagsHub / local `mlruns/` |
| 🗄️ MongoDB Record | `mixtape_automation.runs` collection |
| 📦 ZIP | `/content/mixtape_artifacts_*.zip` |

---
### 🚀 Future Scope
- AI-based genre detection
- Auto-upload to YouTube via API
- Waveform visualizer overlay in video
- User-selectable transition styles
- Streamlit / FastAPI frontend integration